# Hypothesis 1 Testing

_If someone reaches the bottom of a story many times, they're more likely to subscribe to newsletters._

## Clarification

- "bottom" -> scroll depth & need to define what "bottom" is based on data (e.g., histogram)
- "story" -> only look at page views for articles (i.e., `page_is_article` is `True`)
- "many times" -> we're interested in cardinality, which means we should come up with a time window to not favor users who've been around longer. Let's say the time window is X days
    - First, query all users who submits a newsletter form. For each user, look at the distribution of the number of days between their first ever page view and when they submits the newsletter form to get a sense of what X should minimally be.
    - Then:
        - For users who submit a newsletter form: count number of articles they've reached bottom in past X days
        - For users who don't submit a newsletter form: get rolling counts of number of articles each reach bottom in an X-day window, then calculate the average of all those counts
- "more likely to subscribe to newsletters" -> keeping a 0/1 flag of whether a user subscribes to newsletter

## Preprocessing

In [ ]:
from enum import Enum, auto

import matplotlib.pyplot as plt
import pandas as pd
from helpers.pathlib import get_path_site_checkpoint

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow, _StrEnum
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

In [ ]:
# INSTRUCTION: If there's a new field you'd like to add, add its name to this enum
class FieldTemp(_StrEnum):
    DAYS_FROM_EARLIEST_TO_SUBMISSION = auto()
    #     DERIVED_TSTAMP_DT = auto()
    EARLIEST_EVENT_TSTAMP = auto()
    #     EARLIEST_EVENT_TSTAMP_DT = auto()
    LAST_SUBMISSION_TSTAMP = auto()
    #     LAST_SUBMISSION_TSTAMP_DT = auto()
    SUBMITTED = auto()
    USER_SCROLLS_TO_BOTTOM = auto()

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(7, THE_19TH.name))

In [ ]:
# Extract date from timestamp
# def extract_date(df):
#     df = df.copy()
#     df[FieldTemp.DERIVED_TSTAMP_DT] = df[FieldSnowplow.DERIVED_TSTAMP].dt.date
#     return df


# df_afla = extract_date(df_afla)

In [ ]:
# Filter to page views of articles and page views where a newsletter-form submission was recorded
def get_articles_and_submissions_dfs(df, site_name):
    df_articles = df.query(FieldNew.PAGE_IS_ARTICLE)
    df_submissions = df.query(FieldNew.FORM_SUBMIT_IS_NEWSLETTER)
    print(f"Site: {site_name}")
    print(f"Found {df_articles.shape[0]} page views for articles")
    print(f"Found {df_submissions.shape[0]} page views where a newsletter form is submitted")
    print("\n")
    return df_articles, df_submissions


df_afla_articles, df_afla_submissions = get_articles_and_submissions_dfs(df_afla, AFRO_LA.name)

### Define bottom threshold

In [ ]:
# Scroll depth histograms
def plot_scroll_depth_distribution(df, site_name):
    df[FieldNew.SCROLL_DEPTH_MAX].plot.hist(title=f"{site_name} scroll depth dist. ", bins=40)
    plt.show()


plot_scroll_depth_distribution(df_afla_articles, AFRO_LA.name)

In [ ]:
# INSTRUCTION: ADD/UPDATE scroll depth threshold at which to consider as "bottom" here
class ThresholdBottom(float, Enum):
    AFRO_LA = 2 / 3

In [ ]:
# Add a boolean indicating whether a user has reached bottom in an event
def add_field_user_scroll_to_bottom(df, site_name, threshold_bottom):
    df = df.copy()
    df[FieldTemp.USER_SCROLLS_TO_BOTTOM] = df[FieldNew.SCROLL_DEPTH_MAX] >= threshold_bottom
    num_events_bottom = df[FieldTemp.USER_SCROLLS_TO_BOTTOM].sum()
    print(f"Site: {site_name}")
    print(
        f"Found {num_events_bottom} article page views ({num_events_bottom / df.shape[0]:.1%}) "
        + "events in which the reader scrolls to the bottom."
    )
    print("\n")
    return df


df_afla_articles = add_field_user_scroll_to_bottom(df_afla_articles, AFRO_LA.name, ThresholdBottom.AFRO_LA)

### Get all unique users

In [ ]:
# Group events by user and get the date of earliest event for each user
def group_events(df, site_name):
    df_grouped = (
        df.groupby(FieldSnowplow.DOMAIN_USERID)
        .aggregate({FieldSnowplow.DERIVED_TSTAMP: "min"})
        .rename(columns={FieldSnowplow.DERIVED_TSTAMP: FieldTemp.EARLIEST_EVENT_TSTAMP})
    )
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} unique readers")
    print("\n")
    return df_grouped


df_afla_users = group_events(df_afla, AFRO_LA.name)

### Get unique users who sign up

In [ ]:
# Group events by user. If a user has more than one event, take the most recent one to ensure X is
# as big as it can be
def group_submissions(df, site_name):
    df_grouped = (
        df.groupby(FieldSnowplow.DOMAIN_USERID)
        .aggregate({FieldSnowplow.DERIVED_TSTAMP: "max"})
        .rename(columns={FieldSnowplow.DERIVED_TSTAMP: FieldTemp.LAST_SUBMISSION_TSTAMP})
        .merge(df_afla_users[FieldTemp.EARLIEST_EVENT_TSTAMP], left_index=True, right_index=True)
    )
    df_grouped[FieldTemp.SUBMITTED] = True
    df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION] = (
        (df_grouped[FieldTemp.LAST_SUBMISSION_TSTAMP] - df_grouped[FieldTemp.EARLIEST_EVENT_TSTAMP]).dt.total_seconds()
        / 60
        / 60
        / 24
    )
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} readers who signed up for newsletter")
    print(
        f"Smallest window between first event and newsletter sign-up: {df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION].min():.3} days"
    )
    print(
        f"Greatest window between first event and newsletter sign-up: {df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION].max():.3} days"
    )
    print("\n")
    return df_grouped


df_afla_users_submission = group_submissions(df_afla_submissions, AFRO_LA.name)

### Define rolling count window

In [ ]:
class RollingWindow(int, Enum):
    AFRO_LA = 0  # Same day

### Get unique users who read at least one article